# Modelling baselines

In [2]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report
)

import time
import joblib
import os

In [3]:
train_processed = pd.read_csv("../data/processed/train_processed.csv")
val_processed = pd.read_csv("../data/processed/val_processed.csv")
test_processed = pd.read_csv("../data/processed/test_processed.csv")

print("Train:", train_processed.shape)
print("Validation:", val_processed.shape)
print("Test:", test_processed.shape)

Train: (1086991, 12)
Validation: (155226, 12)
Test: (309993, 12)


In [4]:
final_features = [
    "Hemodynamic",
    "Respiratory",
    "Renal_Metabolic",
    "Inflammatory_Hematological",
    "Hepatic_Coagulation",
    "Hour",
    "ICULOS",
    "HospAdmTime",
    "Gender",
    "ICU_Unit"
]

target = "SepsisLabel"

X_train = train_processed[final_features]
y_train = train_processed[target]

X_val = val_processed[final_features]
y_val = val_processed[target]

X_test = test_processed[final_features]
y_test = test_processed[target]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (1086991, 10)
X_val: (155226, 10)
X_test: (309993, 10)


In [5]:
print("Training target distribution:")
print(y_train.value_counts())

print("\nTraining target percentage:")
print(
    y_train.value_counts(normalize=True).mul(100).round(2)
)

Training target distribution:
SepsisLabel
0    1067723
1      19268
Name: count, dtype: int64

Training target percentage:
SepsisLabel
0    98.23
1     1.77
Name: proportion, dtype: float64


In [6]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Negative samples:", negative_count)
print("Positive samples:", positive_count)
print("Class weight:", scale_pos_weight)

Negative samples: 1067723
Positive samples: 19268
Class weight: 55.41431388831223


In [7]:
class_weight="balanced"

In [8]:
def evaluate_model(model, X, y, dataset_name="Validation"):
    
    probabilities = model.predict_proba(X)[:, 1]
    
    predictions = (probabilities >= 0.5).astype(int)

    pr_auc = average_precision_score(y, probabilities)
    roc_auc = roc_auc_score(y, probabilities)

    precision = precision_score(y, predictions, zero_division=0)
    recall = recall_score(y, predictions, zero_division=0)
    f1 = f1_score(y, predictions, zero_division=0)
    accuracy = accuracy_score(y, predictions)

    print(f"\n{dataset_name} Results")
    print("-" * 40)
    print(f"PR-AUC:    {pr_auc:.6f}")
    print(f"ROC-AUC:   {roc_auc:.6f}")
    print(f"Precision:  {precision:.6f}")
    print(f"Recall:     {recall:.6f}")
    print(f"F1:         {f1:.6f}")
    print(f"Accuracy:   {accuracy:.6f}")

    return {
        "PR-AUC": pr_auc,
        "ROC-AUC": roc_auc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Accuracy": accuracy
    }

In [9]:
logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=200,
    solver="lbfgs",
    n_jobs=-1
)

In [11]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_val_imp = pd.DataFrame(
    imputer.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)

X_test_imp = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

In [13]:
start_time = time.time()

logistic_model.fit(X_train_imp, y_train)

logistic_time = time.time() - start_time

print(f"Training time: {logistic_time:.2f} seconds")

c:\Users\bobs9\OneDrive\Desktop\Intern_project\Virtenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


Training time: 22.67 seconds


c:\Users\bobs9\OneDrive\Desktop\Intern_project\Virtenv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [15]:
logistic_val_results = evaluate_model(
    logistic_model,
    X_val_imp,
    y_val,
    "Logistic Regression - Validation"
)


Logistic Regression - Validation Results
----------------------------------------
PR-AUC:    0.073398
ROC-AUC:   0.735228
Precision:  0.042640
Recall:     0.603746
F1:         0.079654
Accuracy:   0.750493


In [16]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_leaf=20,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

start_time = time.time()

rf_model.fit(X_train_imp, y_train)

rf_time = time.time() - start_time

print(f"Random Forest training time: {rf_time:.2f} seconds")

Random Forest training time: 198.59 seconds


In [17]:
rf_val_results = evaluate_model(
    rf_model,
    X_val_imp,
    y_val,
    "Random Forest - Validation"
)


Random Forest - Validation Results
----------------------------------------
PR-AUC:    0.085073
ROC-AUC:   0.781099
Precision:  0.058168
Recall:     0.560519
F1:         0.105399
Accuracy:   0.829835


In [18]:
hgb_model = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    min_samples_leaf=20,
    class_weight="balanced",
    random_state=42
)

start_time = time.time()

hgb_model.fit(X_train, y_train)

hgb_time = time.time() - start_time

print(f"Training time: {hgb_time:.2f} seconds")

Training time: 31.33 seconds


In [19]:
hgb_val_results = evaluate_model(
    hgb_model,
    X_val,
    y_val,
    "HistGradientBoosting - Validation"
)


HistGradientBoosting - Validation Results
----------------------------------------
PR-AUC:    0.083325
ROC-AUC:   0.782045
Precision:  0.053660
Recall:     0.600504
F1:         0.098517
Accuracy:   0.803461


___
___

In [20]:
from sklearn.svm import LinearSVC

In [21]:
svm_model = LinearSVC(
    class_weight="balanced",
    C=1.0,
    max_iter=1000,
    random_state=42
)

start_time = time.time()

svm_model.fit(X_train_imp, y_train)

svm_time = time.time() - start_time

print(f"Linear SVM training time: {svm_time:.2f} seconds")

Linear SVM training time: 8.81 seconds


In [22]:
def evaluate_linear_svm(model, X, y, dataset_name="Validation"):

    scores = model.decision_function(X)

    predictions = (scores >= 0).astype(int)

    pr_auc = average_precision_score(y, scores)
    roc_auc = roc_auc_score(y, scores)

    precision = precision_score(
        y, predictions, zero_division=0
    )

    recall = recall_score(
        y, predictions, zero_division=0
    )

    f1 = f1_score(
        y, predictions, zero_division=0
    )

    accuracy = accuracy_score(
        y, predictions
    )

    print(f"\n{dataset_name} Results")
    print("-" * 40)
    print(f"PR-AUC:    {pr_auc:.6f}")
    print(f"ROC-AUC:   {roc_auc:.6f}")
    print(f"Precision:  {precision:.6f}")
    print(f"Recall:     {recall:.6f}")
    print(f"F1:         {f1:.6f}")
    print(f"Accuracy:   {accuracy:.6f}")

    return {
        "PR-AUC": pr_auc,
        "ROC-AUC": roc_auc,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Accuracy": accuracy
    }

In [23]:
svm_val_results = evaluate_linear_svm(
    svm_model,
    X_val_imp,
    y_val,
    "Linear SVM - Validation"
)


Linear SVM - Validation Results
----------------------------------------
PR-AUC:    0.070716
ROC-AUC:   0.735272
Precision:  0.043701
Recall:     0.592579
F1:         0.081399
Accuracy:   0.760813


In [24]:
baseline_results = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        **logistic_val_results,
        "Training_Time_Seconds": logistic_time
    },
    {
        "Model": "Random Forest",
        **rf_val_results,
        "Training_Time_Seconds": rf_time
    },
    {
        "Model": "HistGradientBoosting",
        **hgb_val_results,
        "Training_Time_Seconds": hgb_time
    },
    {
        "Model": "Linear SVM",
        **svm_val_results,
        "Training_Time_Seconds": svm_time
    }
])

baseline_results

,Model,PR-AUC,ROC-AUC,Precision,Recall,F1,Accuracy,Training_Time_Seconds
0,Logistic Regression,0.073398,0.735228,0.042640,0.603746,0.079654,0.750493,22.672658
1,Random Forest,0.085073,0.781099,0.058168,0.560519,0.105399,0.829835,198.589184
2,HistGradientBoosting,0.083325,0.782045,0.053660,0.600504,0.098517,0.803461,31.328793
3,Linear SVM,0.070716,0.735272,0.043701,0.592579,0.081399,0.760813,8.807694


___
___

In [27]:
print("Logistic Regression features:", logistic_model.n_features_in_)
print("SVM features:", svm_model.n_features_in_)

print("\nLogistic coefficients shape:", logistic_model.coef_.shape)
print("SVM coefficients shape:", svm_model.coef_.shape)

print("\nLogistic classes:", logistic_model.classes_)
print("SVM classes:", svm_model.classes_)

Logistic Regression features: 10
SVM features: 10

Logistic coefficients shape: (1, 10)
SVM coefficients shape: (1, 10)

Logistic classes: [0 1]
SVM classes: [0 1]


In [26]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

joblib.dump(logistic_model, "../models/logistic_regression_baseline.pkl")
joblib.dump(rf_model, "../models/random_forest_baseline.pkl")
joblib.dump(hgb_model, "../models/hist_gradient_boosting_baseline.pkl")
joblib.dump(svm_model, "../models/linear_svm_baseline.pkl")

print("Baseline models and preprocessing objects saved.")

Baseline models and preprocessing objects saved.
